In [0]:
SELECT *
FROM workspace.sqlproject.gold_fact_sales
LIMIT 100

In [0]:
SELECT *
FROM workspace.sqlproject.gold_dim_products
LIMIT 100

In [0]:
SELECT *
FROM workspace.sqlproject.gold_dim_customers
LIMIT 100

**Part 1: Getting to Know the Customers**

1.1 Start by listing all customers from Australia.

1.2 The marketing team is interested in customer demographics. Show the number of customers grouped by gender.

1.3 Create a column that segments customers into age groups:

●	<30 = Young

●	30–50 = Adult

●	>50 = Senior

Then, count how many customers fall into each group.

In [0]:
SELECT *
FROM workspace.sqlproject.gold_dim_customers
WHERE country = "Australia"

In [0]:
SELECT gender, COUNT(*) AS customer_count
FROM  workspace.sqlproject.gold_dim_customers
GROUP BY gender;

In [0]:
SELECT customer_key, customer_id, first_name, last_name, birthdate, YEAR(birthdate) AS birth_year, 
2025 - YEAR(birthdate) AS age,
CASE 
WHEN 2025 - YEAR(birthdate)<30 THEN "Young"
WHEN 2025 - YEAR(birthdate)<50 THEN "Adult"
ELSE "Senior"
END AS age_group
FROM workspace.sqlproject.gold_dim_customers

In [0]:
SELECT
   SUM(CASE WHEN 2025 - YEAR(birthdate)<30 THEN 1 ELSE 0 END) AS below_30,
   SUM (CASE WHEN 2025 - YEAR(birthdate) BETWEEN 30 AND 49 THEN 1 ELSE 0 END) AS 30_50,
   SUM (CASE WHEN 2025 - YEAR(birthdate) > 50 THEN 1 ELSE 0 END) AS above_50
FROM workspace.sqlproject.gold_dim_customers

**Part 2: Exploring Sales Over Time**

2.1 List all sales orders that were made in March 2013.

2.2 Calculate the total revenue and total quantity for each month in 2013.

2.3 Find out which month in 2013 had the highest total sales amount.

In [0]:
SELECT *
FROM workspace.sqlproject.gold_fact_sales
WHERE YEAR(order_date) = 2013 AND MONTH(order_date) = 3

In [0]:
SELECT MONTH(order_date) AS month, SUM(quantity*price) AS total_revenue, SUM(quantity) AS total_quantity
FROM workspace.sqlproject.gold_fact_sales
WHERE YEAR(order_date)=2013
GROUP BY MONTH(order_date)
ORDER BY MONTH(order_date) ASC ;

In [0]:
SELECT MONTH(order_date) AS month, SUM(sales_amount)
FROM workspace.sqlproject.gold_fact_sales
WHERE YEAR(order_date)=2013
GROUP BY 1
ORDER BY 2 DESC
LIMIT 1;

**Part 3: Products and Categories**

3.1 Show all products that belong to the Bikes category.

3.2 Join the sales table with products and calculate the total sales amount per product name.

3.3 For each product category, calculate its total revenue.

3.4 Which five products generated the highest revenue?

In [0]:
SELECT *
FROM workspace.sqlproject.gold_dim_products
WHERE category = "Bikes";

In [0]:
SELECT p.product_name, SUM(s.sales_amount)
FROM workspace.sqlproject.gold_dim_products p
LEFT JOIN workspace.sqlproject.gold_fact_sales s
  ON p.product_key = s.product_key
GROUP BY p.product_name
ORDER BY 2;

In [0]:
SELECT p.category, SUM(s.sales_amount)
FROM workspace.sqlproject.gold_fact_sales s
LEFT JOIN workspace.sqlproject.gold_dim_products p
  ON p.product_key = s.product_key
GROUP BY p.category
ORDER BY 2 DESC;

In [0]:
SELECT p.product_name, SUM(s.sales_amount)
FROM workspace.sqlproject.gold_dim_products p
LEFT JOIN workspace.sqlproject.gold_fact_sales s
  ON p.product_key = s.product_key
GROUP BY p.product_name
ORDER BY 2 DESC
LIMIT 5;

**Part 4: Customer & Product Performance**

4.1 Join the sales and customer tables to find the top 5 customers by spending.

4.2 For each gender, calculate the total sales revenue.

4.3 Segment orders into two types:

●	HighValueOrder if sales_amount > 1000

●	RegularOrder otherwise

Then, count how many orders fall into each type.

In [0]:
SELECT c.customer_number, c.first_name, c.last_name, SUM(sales_amount) AS spending
FROM workspace.sqlproject.gold_dim_customers c
LEFT JOIN workspace.sqlproject.gold_fact_sales s
  ON c.customer_key = s.customer_key
GROUP BY c.customer_number, c.first_name, c.last_name
ORDER BY 4 DESC
LIMIT 5;

In [0]:
SELECT c.gender, SUM(s.quantity*s.price) as total_revenue
FROM workspace.sqlproject.gold_dim_customers c
LEFT JOIN workspace.sqlproject.gold_fact_sales s
  ON c.customer_key = s.customer_key
GROUP BY c.gender;

In [0]:
SELECT
  SUM (CASE WHEN sales_amount > 1000 THEN 1 ELSE 0 END) AS above_1000,
  SUM (CASE WHEN sales_amount <= 1000 THEN 1 ELSE 0 END) AS below_1000
FROM workspace.sqlproject.gold_fact_sales;

**Part 5: Contribution & Insights**

5.1 Calculate the contribution (%) of each product category to the company's overall revenue.

5.2 Use a subquery to find customers whose total spending is above the average customer spending.

5.3 Use a subquery to find products whose revenue is above the average product revenue.

In [0]:
SELECT p.category, SUM(s.sales_amount) AS category_revenue,
ROUND (SUM(s.sales_amount)*100 / (SELECT SUM(s.sales_amount) FROM workspace.sqlproject.gold_fact_sales s),2) AS contribution
FROM workspace.sqlproject.gold_fact_sales s
LEFT JOIN workspace.sqlproject.gold_dim_products p
  ON p.product_key = s.product_key
GROUP BY 1

In [0]:
SELECT customer_number, first_name, last_name, SUM(sales_amount) AS Total_spending
FROM workspace.sqlproject.gold_fact_sales s
LEFT JOIN workspace.sqlproject.gold_dim_customers c
ON s.customer_key = c.customer_key
GROUP BY 1,2,3
HAVING SUM(sales_amount) > (
  SELECT AVG(Total_spending)
  FROM (
    SELECT customer_number, first_name, last_name, SUM(sales_amount) AS Total_spending
    FROM workspace.sqlproject.gold_fact_sales s
    LEFT JOIN workspace.sqlproject.gold_dim_customers c
    ON s.customer_key = c.customer_key
    GROUP BY 1,2,3
  )
)

In [0]:
SELECT p.product_name, SUM(sales_amount) AS Total_revenue
FROM workspace.sqlproject.gold_fact_sales s
LEFT JOIN workspace.sqlproject.gold_dim_products p
ON p.product_key = s.product_key
GROUP BY 1
HAVING SUM(sales_amount) > (
  SELECT AVG(Total_revenue)
  FROM (
    SELECT p.product_name, SUM(sales_amount) AS Total_revenue
    FROM workspace.sqlproject.gold_fact_sales s
    LEFT JOIN workspace.sqlproject.gold_dim_products p
    ON p.product_key = s.product_key
    GROUP BY 1
  )
)

**Part 6: Extra Tasks**

6.1 The manager wants a list that combines all customer first names and product names into one single list.

6.2 List all customer IDs who either made a purchase in 2013 OR whose country is Australia.


In [0]:
SELECT c.first_name AS name
FROM workspace.sqlproject.gold_dim_customers c
UNION
SELECT p.product_name AS name
FROM workspace.sqlproject.gold_dim_products p

In [0]:
SELECT c.customer_number
FROM workspace.sqlproject.gold_fact_sales s
LEFT JOIN workspace.sqlproject.gold_dim_customers c
ON s.customer_key = c.customer_key
WHERE YEAR(order_date) = 2013

UNION

SELECT c.customer_number
FROM workspace.sqlproject.gold_dim_customers c
WHERE country = 'Australia';